- Optimizing Nueral Network

In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [55]:
torch.manual_seed(42)

In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Divice is {device}")

Divice is cuda


In [57]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
dataset_path = "/content/drive/MyDrive/DataSets/fashion-mnist_train.csv"

In [59]:
df = pd.read_csv(dataset_path)

In [60]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [61]:
X = df.drop(columns="label")
y = df["label"]

In [62]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [63]:
X_train = X_train/255.0
X_test = X_test/255.0

In [64]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features.to_numpy(),dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(),dtype=torch.long)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [65]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [66]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [67]:
class MyANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,10)
        )
    
    def forward(self,features):
        return self.model(features)
            

In [68]:
learning_rate = 0.1
epochs = 100

In [69]:
# instantiating model
model = MyANN(X_train.shape[1])
model = model.to(device)

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(),lr=learning_rate)

In [70]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # forward pass
        outputs = model(batch_features)

        # calculate loss
        loss = criterion(outputs,batch_labels)

        # back pass
        optimizer.zero_grad()
        loss.backward()

        # gradients update
        optimizer.step()

        # total epoch loss
        total_epoch_loss = total_epoch_loss + loss.item()

    # avg loss
    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

Epoch: 1, Loss: 0.7371663815627495
Epoch: 2, Loss: 0.4531613422185183
Epoch: 3, Loss: 0.4006250313470761
Epoch: 4, Loss: 0.3734306193590164
Epoch: 5, Loss: 0.3491697212457657
Epoch: 6, Loss: 0.3292420780013005
Epoch: 7, Loss: 0.315441799821953
Epoch: 8, Loss: 0.3052361440410217
Epoch: 9, Loss: 0.293991380016009
Epoch: 10, Loss: 0.2822909744580587
Epoch: 11, Loss: 0.27608789296944936
Epoch: 12, Loss: 0.26816045936445393
Epoch: 13, Loss: 0.25673703008765975
Epoch: 14, Loss: 0.25137261414776246
Epoch: 15, Loss: 0.24626374938587348
Epoch: 16, Loss: 0.2391931854188442
Epoch: 17, Loss: 0.23763653363411624
Epoch: 18, Loss: 0.22890330864302813
Epoch: 19, Loss: 0.22451584434633454
Epoch: 20, Loss: 0.2206133975063761
Epoch: 21, Loss: 0.2171072883984695
Epoch: 22, Loss: 0.2097812600657344
Epoch: 23, Loss: 0.2066514995911469
Epoch: 24, Loss: 0.20255969987499217
Epoch: 25, Loss: 0.19668218014140923
Epoch: 26, Loss: 0.19138314977909127
Epoch: 27, Loss: 0.19218520286958665
Epoch: 28, Loss: 0.19025653

In [71]:
# set model to eval mode
model.eval()

MyANN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [72]:
# evaluation code for test data
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

    print(correct/total)
    


0.8895


In [73]:
# evaluation code for training
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in train_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

    print(correct/total)
    

0.9778541666666667
